# SR1.5 feasibility scorer — simplified

**Goal:** take indicator files in, produce per-city ranked actions out. One step per cell, no hidden state.

## The 7 steps

1. Setup — paths and parameters
2. Load inputs — 4 CSV files
3. Reference tables — option families and quintile-to-capacity mapping
4. Build the cell bridge — one row per (option × dimension × indicator × city_bridge); **writes to disk**
5. Define `score_one(action_id, locode)` — scoring function; test on one example
6. Score every action for every city
7. Save outputs

## The scoring math (one line per cell)

For each (action, city), for each SR1.5 A/C cell:

```
sr15_signal       = +1 if C (supportive)  |  -1 if A (barrier)
city_capacity     = QUINT[city's bucket for the bridge's indicator]   # 0.0 - 1.0
bridge_contrib    = bridge_sign × (2 × city_capacity - 1)              # -1 to +1
cell_score        = (sr15_signal + mean(bridge_contribs)) / 2          # if bridges exist
cell_score_01     = (cell_score + 1) / 2                                # 0.0 - 1.0
```

Raw action score = mean over dimensions of (mean cell_score_01 within each dimension).

Final action score shrinks the raw score toward 0.5 by the action's `match_strength` (mapping-confidence weight):

```
strength_weight   = STRENGTH_WEIGHT[match_strength]    # direct=1.0, partial=0.95, cross_cutting=0.9, weak=0.85, no_match=0.0
action_score      = 0.5 + strength_weight × (raw_score - 0.5)
```

This tempers the headline score where the SR1.5 option only loosely describes the action (e.g. waste actions mapped to *Reduced food wastage* because SR1.5 has no general-waste option). The per-dimension breakdown stays raw so it still reflects the underlying IPCC+city evidence.

## Step 1 — Setup

Paths and parameters. Change `CITY_INDICATORS_PATH` to score a different set of cities.

In [73]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()             # notebook lives in releases/2018/
DATA = ROOT / 'data'
OUT  = ROOT / 'sample'

ACTIONS_PATH           = DATA / 'actions_to_sr15_mapping.csv'
PER_CELL_PATH          = DATA / 'sr15_feasibility_per_cell.csv'
BRIDGES_PATH           = DATA / 'sr15_indicator_to_city_indicator.csv'
CITY_INDICATORS_PATH   = OUT  / 'test_cities_indicators.csv'

# Derived (written by this notebook)
CELL_BRIDGE_PATH       = DATA / 'sr15_cell_bridge.csv'
RANKED_ACTIONS_PATH    = OUT  / 'test_cities_ranked_actions_by_locode.csv'
RANKED_PER_CELL_PATH   = OUT  / 'test_cities_ranked_actions_per_cell.csv'

# Mapping-confidence weights. The headline action score shrinks toward 0.5 by this weight:
#     action_score = 0.5 + STRENGTH_WEIGHT[match_strength] × (raw_score - 0.5)
# Rationale: a `weak` mapping borrows the SR1.5 option prior less reliably than a `direct` mapping,
# so its score should hedge toward neutral. Direct gets full weight; no_match collapses to 0.5.
STRENGTH_WEIGHT = {
    'direct':        1.00,
    'partial':       0.95,
    'cross_cutting': 0.90,
    'weak':          0.85,
    'no_match':      0.00,
}

## Step 2 — Load inputs

Four CSV files. Each is one logical input.

In [74]:
actions     = pd.read_csv(ACTIONS_PATH)              # action_id → sr15_options (102 actions)
per_cell    = pd.read_csv(PER_CELL_PATH)              # SR1.5 priors: option × dimension × indicator → A/B/C/NA/NE/LE
bridges     = pd.read_csv(BRIDGES_PATH)               # SR1.5 indicator + scope → city_indicator + sign
city_df     = pd.read_csv(CITY_INDICATORS_PATH)       # city × attribute_type → attribute_category

print(f'actions      : {len(actions):>5d} rows  ({actions.action_id.nunique()} unique action_ids)')
print(f'per_cell     : {len(per_cell):>5d} rows  ({per_cell.option.nunique()} SR1.5 options × {per_cell.dimension.nunique()} dimensions)')
print(f'bridges      : {len(bridges):>5d} rows  ({bridges.city_indicator.nunique()} city_indicators)')
print(f'city_df      : {len(city_df):>5d} rows  ({city_df.locode.nunique()} cities × {city_df.attribute_type.nunique()} indicators)')

actions      :   102 rows  (102 unique action_ids)
per_cell     :   547 rows  (27 SR1.5 options × 6 dimensions)
bridges      :    54 rows  (24 city_indicators)
city_df      :   330 rows  (12 cities × 34 indicators)


## Step 3 — Reference tables

Two small lookup dicts:

- `OPTION_FAMILY`: every SR1.5 option belongs to one scope family (`energy_supply`, `transport`, `buildings`, etc.). Bridges with `scope=all` apply to every option; bridges with a specific scope only apply to options in that family.
- `QUINT`: the city's categorical bucket maps to a capacity number between 0 and 1. `very low` → 0, `very high` → 1.

In [75]:
OPTION_FAMILY = {
    # energy supply
    'Wind (on-shore & off-shore)':'energy_supply','Solar PV':'energy_supply','Bioenergy':'energy_supply',
    'Electricity storage':'energy_supply','Power sector CCS':'energy_supply','Nuclear energy':'energy_supply',
    'Smart grids':'energy_supply','BECCS':'energy_supply','DACCS':'energy_supply',
    # waste / dietary
    'Reduced food wastage':'waste','Dietary shifts':'waste',
    # nature-based
    'Sustainable intensification':'sustainable_intensification',  # split out so pasture/cropland bridges target livestock/crop options without firing on afforestation
    'Afforestation & reforestation':'nbs',
    'Soil carbon sequestration & biochar':'nbs','Enhanced weathering':'nbs',
    # cross-cutting
    'Land-use & urban planning':'cross_cutting',
    # transport
    'Electric cars and buses':'transport','Sharing schemes':'transport','Public transport':'transport',
    'Non-motorised transport':'transport','Aviation & shipping':'transport',
    # buildings
    'Efficient appliances':'buildings','Low/zero-energy buildings':'buildings',
    # industrial
    'Energy efficiency':'industrial','Bio-based & circularity':'industrial',
    'Electrification & hydrogen':'industrial','Industrial CCS':'industrial',
}

QUINT = {'very low':0.0, 'low':0.25, 'medium':0.5, 'high':0.75, 'very high':1.0}

print(f'OPTION_FAMILY covers {len(OPTION_FAMILY)} SR1.5 options across {len(set(OPTION_FAMILY.values()))} scope families')
print(f'Scope families: {sorted(set(OPTION_FAMILY.values()))}')
print(f'\nQUINT mapping: {QUINT}')

OPTION_FAMILY covers 27 SR1.5 options across 8 scope families
Scope families: ['buildings', 'cross_cutting', 'energy_supply', 'industrial', 'nbs', 'sustainable_intensification', 'transport', 'waste']

QUINT mapping: {'very low': 0.0, 'low': 0.25, 'medium': 0.5, 'high': 0.75, 'very high': 1.0}


## Step 4 — Build the cell bridge

Walk every A/C cell in `per_cell` and attach the city bridges that apply (matching `indicator` + scope ∈ {`all`, `option_family`}). Cells with code B/NE/LE/NA get no city adjustment (per the A/C rule in `review.md`).

**Writes `sr15_cell_bridge.csv`.** This file is the scorer's working bridge table. If you change `sr15_indicator_to_city_indicator.csv` upstream, you must re-run this cell.

In [76]:
# Only keep bridges that reference a real city_indicator (skip option-level constants)
real_bridges = bridges[bridges.city_indicator != '(option-level constant)']

rows = []
for _, cell in per_cell.iterrows():
    option, indicator, code, dimension = cell.option, cell.indicator, cell.code, cell.dimension
    family = OPTION_FAMILY.get(option, '?')

    # B / NE / LE / NA — no directional evidence, no city adjustment
    if code not in ('A', 'C'):
        rows.append({'option':option,'dimension':dimension,'indicator':indicator,'sr15_code':code,
                     'option_family':family,'city_indicator':f'(SR1.5 code {code}: no directional evidence)',
                     'sign':0,'scope':''})
        continue

    # Find bridges that match (indicator, scope ∈ {all, family})
    matches = real_bridges[(real_bridges.sr15_indicator == indicator) &
                           (real_bridges.scope.isin(['all', family]))]

    if matches.empty:
        rows.append({'option':option,'dimension':dimension,'indicator':indicator,'sr15_code':code,
                     'option_family':family,'city_indicator':'(no scope-matching bridge)',
                     'sign':0,'scope':''})
    else:
        for _, b in matches.iterrows():
            rows.append({'option':option,'dimension':dimension,'indicator':indicator,'sr15_code':code,
                         'option_family':family,'city_indicator':b.city_indicator,
                         'sign':int(b.sign),'scope':b.scope})

cell_bridge = pd.DataFrame(rows)
cell_bridge.to_csv(CELL_BRIDGE_PATH, index=False)

active = (cell_bridge.sign != 0).sum()
print(f'Wrote {CELL_BRIDGE_PATH.name}: {len(cell_bridge)} rows ({active} with active city bridge)')
print(f'\nBreakdown by scope:')
print(cell_bridge[cell_bridge.sign != 0].scope.value_counts().to_string())

Wrote sr15_cell_bridge.csv: 598 rows (127 with active city bridge)

Breakdown by scope:
scope
all                            77
buildings                      11
energy_supply                  10
transport                      10
nbs                            10
industrial                      7
sustainable_intensification     2


## Step 5 — Score one (action, city) pair

The scoring function. Pure — no globals beyond the input dataframes.

In [77]:
def score_one(action_id, locode, actions, per_cell, cell_bridge, city_df):
    """Return (overall_score, cells_dataframe) for one action × one city."""

    # 1. Get the SR1.5 options this action maps to
    action_row = actions[actions.action_id == action_id].iloc[0]
    options = [o for o in action_row.sr15_options.split('|') if not o.startswith('(')]
    if not options:
        return None, pd.DataFrame()

    # 2. Get the city's bucket for each indicator the city has data for
    city_data = city_df[city_df.locode == locode]
    city_capacity = {r.attribute_type: QUINT[r.attribute_category] for _, r in city_data.iterrows() if r.attribute_category in QUINT}

    # 3. Pull just the A/C cells for this action's options
    action_cells = per_cell[per_cell.option.isin(options) & per_cell.code.isin(['A', 'C'])]
    if action_cells.empty:
        return None, pd.DataFrame()

    # 4. Score each cell
    cell_scores = []
    for _, cell in action_cells.iterrows():
        sr15_signal = +1 if cell.code == 'C' else -1

        bridges_for_cell = cell_bridge[(cell_bridge.option == cell.option) &
                                       (cell_bridge.indicator == cell.indicator) &
                                       (cell_bridge.sign != 0)]

        contribs = [int(b.sign) * (2 * city_capacity[b.city_indicator] - 1)
                    for _, b in bridges_for_cell.iterrows()
                    if b.city_indicator in city_capacity]

        if contribs:
            cell_score = (sr15_signal + sum(contribs) / len(contribs)) / 2
        else:
            cell_score = sr15_signal

        cell_scores.append({'option':cell.option, 'dim':cell.dimension, 'indicator':cell.indicator,
                            'code':cell.code, 'cell_score_01':(cell_score + 1) / 2})

    cells = pd.DataFrame(cell_scores)
    # Overall score = mean of (mean cell_score_01 within each dimension)
    overall = cells.groupby('dim')['cell_score_01'].mean().mean()
    return overall, cells

# Smoke test on one action × one city
score, cells = score_one('c40_0015', 'CL ZAL', actions, per_cell, cell_bridge, city_df)
print(f"c40_0015 (Retrofit residential buildings) for Valdivia: {score:.3f}")
print(f"\nPer-dimension breakdown:")
print(cells.groupby('dim')['cell_score_01'].mean().round(3).to_string())

c40_0015 (Retrofit residential buildings) for Valdivia: 0.913

Per-dimension breakdown:
dim
economic          1.000
environmental     1.000
geophysical       0.750
institutional     1.000
socio_cultural    0.750
technological     0.979


## Step 6 — Score every action for every city

Loop over (city × action). Skip cells that produce `None` (action has no scoreable SR1.5 mapping).

In [78]:
cities = city_df[['locode', 'region_name']].drop_duplicates().sort_values('locode').reset_index(drop=True)
print(f'Scoring {actions.action_id.nunique()} actions × {len(cities)} cities = {actions.action_id.nunique() * len(cities)} pairs')

ranked_rows = []
per_cell_rows = []

for _, city in cities.iterrows():
    locode, region = city.locode, city.region_name
    city_data = city_df[city_df.locode == locode]
    city_cat = {r.attribute_type: r.attribute_category for _, r in city_data.iterrows()}
    city_capacity = {r.attribute_type: QUINT[r.attribute_category] for _, r in city_data.iterrows() if r.attribute_category in QUINT}

    for aid in actions.action_id:
        score, cells = score_one(aid, locode, actions, per_cell, cell_bridge, city_df)
        if score is None:
            continue

        a = actions[actions.action_id == aid].iloc[0]
        dim_scores = cells.groupby('dim')['cell_score_01'].mean().to_dict()

        # Apply mapping-confidence shrinkage to the headline score.
        # Per-dimension scores stay raw (they describe IPCC+city evidence, independent of mapping confidence).
        strength_weight  = STRENGTH_WEIGHT.get(a.match_strength, 0.0)
        adjusted_score   = 0.5 + strength_weight * (score - 0.5)

        ranked_rows.append({
            'locode':            locode,
            'region_name':       region,
            'action_id':         aid,
            'action_name':       a.action_name,
            'primary_outcome':   a.primary_outcome,
            'primary_channel':   a.primary_channel,
            'sr15_options':      a.sr15_options,
            'match_strength':    a.match_strength,
            'strength_weight':   strength_weight,
            'n_ac_cells':        len(cells),
            'n_dims_scored':     len(dim_scores),
            'econ':              round(dim_scores.get('economic',       float('nan')), 3),
            'tech':              round(dim_scores.get('technological',  float('nan')), 3),
            'inst':              round(dim_scores.get('institutional',  float('nan')), 3),
            'soc':               round(dim_scores.get('socio_cultural', float('nan')), 3),
            'env':               round(dim_scores.get('environmental',  float('nan')), 3),
            'geo':               round(dim_scores.get('geophysical',    float('nan')), 3),
            'raw_score':         round(score, 3),
            'score':             round(adjusted_score, 3),
        })

        # Per-cell detail rows for the diagnostic file
        for _, c in cells.iterrows():
            bridges_for_cell = cell_bridge[(cell_bridge.option == c.option) &
                                           (cell_bridge.indicator == c.indicator) &
                                           (cell_bridge.sign != 0)]
            base = {'locode':locode, 'action_id':aid, 'sr15_option':c.option, 'dimension':c.dim,
                    'indicator':c.indicator, 'sr15_code':c.code, 'cell_score_01':round(c.cell_score_01, 3)}
            if bridges_for_cell.empty:
                per_cell_rows.append({**base, 'city_indicator':'(no bridge)', 'sign':0,
                                      'city_bucket':'', 'city_capacity':'', 'indicator_contribution':''})
            else:
                for _, b in bridges_for_cell.iterrows():
                    ci = b.city_indicator
                    if ci in city_capacity:
                        cap = city_capacity[ci]
                        bucket = city_cat[ci]
                        contrib = round(int(b.sign) * (2 * cap - 1), 3)
                    else:
                        cap, bucket, contrib = '', '(no city data)', ''
                    per_cell_rows.append({**base, 'city_indicator':ci, 'sign':int(b.sign),
                                          'city_bucket':bucket, 'city_capacity':cap,
                                          'indicator_contribution':contrib})

print(f'Done: {len(ranked_rows)} (action × city) pairs scored, {len(per_cell_rows)} per-cell diagnostic rows')

Scoring 102 actions × 12 cities = 1224 pairs
Done: 1032 (action × city) pairs scored, 11940 per-cell diagnostic rows


## Step 7 — Sort and save

Per-city ranking (descending by score) and per-cell diagnostic. The diagnostic file is the one you check when you want to see WHY a particular action got its score.

In [79]:
ranked = (pd.DataFrame(ranked_rows)
          .sort_values(['locode', 'score'], ascending=[True, False])
          .reset_index(drop=True))
ranked.to_csv(RANKED_ACTIONS_PATH, index=False)

per_cell_df = pd.DataFrame(per_cell_rows)
per_cell_df.to_csv(RANKED_PER_CELL_PATH, index=False)

print(f'Wrote {RANKED_ACTIONS_PATH.name}: {len(ranked)} rows')
print(f'Wrote {RANKED_PER_CELL_PATH.name}: {len(per_cell_df)} rows')
print(f'\nTop 5 actions per city:')
for loc in ranked.locode.unique():
    top5 = ranked[ranked.locode == loc].head(5)
    print(f'\n  {loc}:')
    for _, r in top5.iterrows():
        print(f'    {r.score:.3f}  {r.action_name[:60]}')

Wrote test_cities_ranked_actions_by_locode.csv: 1032 rows
Wrote test_cities_ranked_actions_per_cell.csv: 11940 rows

Top 5 actions per city:

  CL CRR:
    0.962  Develop integrated renewable energy planning and solar–batte
    0.962  Expand solar energy generation on municipal facilities and p
    0.962  Stimulate solar energy production in industrial facilities
    0.958  Improve heat and energy recovery
    0.958  Deploy cogeneration systems in industries

  CL FUT:
    0.944  Develop integrated renewable energy planning and solar–batte
    0.944  Expand solar energy generation on municipal facilities and p
    0.944  Stimulate solar energy production in industrial facilities
    0.922  Adopt hybrid renewable energy systems (solar-wind)
    0.917  Improve heat and energy recovery

  CL LAN:
    0.958  Improve heat and energy recovery
    0.958  Develop integrated renewable energy planning and solar–batte
    0.958  Deploy cogeneration systems in industries
    0.958  Expand solar en

## Step 8 — Full provenance chain (one CSV view of the entire scoring linkage)

Walks from action → IPCC SR1.5 mitigation option → SR1.5 dimension+indicator → bridge → city-level indicator in a single flat table. One row per `(action × mitigation_option × dimension × global_indicator × city_indicator)` tuple.

**Output schema (matches the canonical chain view):**

| column | meaning |
|---|---|
| `action_id` | City action being scored |
| `global_mitigation_option` | IPCC SR1.5 mitigation option the action maps to |
| `action_mapping_strength` | direct / partial / weak / cross_cutting / no_match |
| `strength_weight` | Numeric shrinkage weight from `STRENGTH_WEIGHT` (1.00 / 0.95 / 0.90 / 0.85 / 0.00) — downstream consumers apply `action_score = 0.5 + strength_weight × (raw_score - 0.5)` |
| `option_family` | Sector grouping (energy_supply, transport, buildings, industrial, nbs, etc.) |
| `feasibility_dimension` | One of the 6 SR1.5 dimensions (economic, technological, institutional, socio_cultural, environmental, geophysical) |
| `global_indicator` | The SR1.5 indicator within the dimension (cost-effectiveness, technical_scalability, etc.) |
| `global_verdict_code` | IPCC's cell code: A, B, C, NA, NE, LE |
| `global_verdict_description` | Plain-text translation: barrier / neutral / supportive / not applicable / no evidence / limited evidence |
| `city_indicator` | The local data point that adjusts this cell (NULL if no bridge applies) |
| `city_indicator_direction` | positive / negative (NULL if no bridge) |
| `city_family_scope` | The sector scope the bridge applies to (NULL if no bridge) |
| `interpretation` | One-sentence explanation of what this row contributes (always populated, including for no-bridge rows) |

City-side columns are NULL when there is no real bridge (either because the IPCC verdict is B/NE/LE/NA, or because no scope-matching bridge has been defined). The `interpretation` column always carries the explanation.

**This step writes two files:**

- `data/scoring_chain.csv` — full 14-column schema with `strength_weight` and `country_code`. Used by the notebook and any downstream consumer that wants the new fields.
- `sample/scoring_chain_full.csv` — legacy 12-column schema. Consumed by the DB chain-table importer that loads `modelled.action_mitigation_feasibility_chain`. Keep them in lockstep — never edit one without re-running this cell.

**Use this to answer:**
- Why did action X score the way it did? → filter by `action_id`
- Which actions does indicator Y influence? → filter by `city_indicator`
- What's the IPCC prior for option Z's geophysical dimension? → filter by `global_mitigation_option` + `feasibility_dimension`
- Where could new bridges add discriminating signal? → `city_indicator IS NULL` and `global_verdict_code IN ('A','C')`

In [84]:
# Optional: pass a city locode to populate per-row local-value columns
SAMPLE_LOCODE = None   # set to "CL ZAL" (or any locode) to add sample_*_bucket/value/capacity columns

# 1. Explode actions → IPCC SR1.5 mitigation options (one row per action × option pair)
action_x_option = []
for _, a in actions.iterrows():
    options = [o.strip() for o in str(a.sr15_options).split("|") if not o.strip().startswith("(")]
    for opt in options:
        action_x_option.append({
            "action_id":                a.action_id,
            "global_mitigation_option": opt,
            "action_mapping_strength":  a.match_strength,
            "strength_weight":          STRENGTH_WEIGHT.get(a.match_strength, 0.0),
        })
action_x_option = pd.DataFrame(action_x_option)

# 2. Join to cell_bridge — gives one row per (action × option × dim × indicator × bridge)
chain = action_x_option.merge(
    cell_bridge.rename(columns={"option": "global_mitigation_option"}),
    on="global_mitigation_option",
    how="left",
)

# 3. Rename canonical columns
chain = chain.rename(columns={
    "dimension": "feasibility_dimension",
    "indicator": "global_indicator",
    "sr15_code": "global_verdict_code",
    "scope":     "city_family_scope",
})

# 4. Translate IPCC verdict code → human description
VERDICT_DESC = {
    "A":  "barrier",
    "B":  "neutral",
    "C":  "supportive",
    "NA": "not applicable",
    "NE": "no evidence",
    "LE": "limited evidence",
}
chain["global_verdict_description"] = chain["global_verdict_code"].map(VERDICT_DESC).fillna(chain["global_verdict_code"])

# 5. Translate bridge direction sign → text
chain["city_indicator_direction"] = chain["sign"].map({1: "positive", -1: "negative"}).fillna("no directional bridge")

# 6. Build the interpretation column (always populated)
def interpret(row):
    code = str(row.global_verdict_code)
    desc = VERDICT_DESC.get(code, code)
    ci   = str(row.city_indicator) if pd.notna(row.city_indicator) else ""

    if code in ("B", "NE", "LE", "NA"):
        return (f"IPCC verdict is '{desc}' for this indicator — provides no directional evidence; "
                f"the cell contributes 0 to the dimension score.")
    if ci.startswith("(no scope-matching"):
        return (f"IPCC verdict is '{desc}' but no city-level data bridge is defined for "
                f"'{row.global_indicator}' at sector scope '{row.option_family}'; "
                f"the IPCC prior is used unmodified.")
    if row.sign == 1:
        return (f"Cities with higher {ci} have stronger feasibility for "
                f"'{row.global_indicator}' ({row.feasibility_dimension} dimension). "
                f"Bridge applies to '{row.city_family_scope}' sector. IPCC base verdict: {desc}.")
    if row.sign == -1:
        return (f"Cities with higher {ci} have weaker feasibility for "
                f"'{row.global_indicator}' ({row.feasibility_dimension} dimension). "
                f"Bridge applies to '{row.city_family_scope}' sector. IPCC base verdict: {desc}.")
    return f"No directional adjustment available for this cell. IPCC base verdict: {desc}."

chain["interpretation"] = chain.apply(interpret, axis=1)

# 7. NULL out city-side columns where there is no real bridge
#    (placeholder strings like "(SR1.5 code B: ...)" or "(no scope-matching bridge)" mean no bridge applies)
no_bridge_mask = chain["city_indicator"].fillna("").astype(str).str.startswith("(")
chain.loc[no_bridge_mask, ["city_indicator", "city_indicator_direction", "city_family_scope"]] = None

# 8. Optional: attach sample-city values + capacity for a chosen locode
if SAMPLE_LOCODE:
    sample = city_df[city_df.locode == SAMPLE_LOCODE]
    city_bucket = dict(zip(sample.attribute_type, sample.attribute_category))
    city_value  = dict(zip(sample.attribute_type, sample.attribute_value))
    QUINT = {"very low": 0.0, "low": 0.25, "medium": 0.5, "high": 0.75, "very high": 1.0}
    suffix = SAMPLE_LOCODE.replace(" ", "_")
    chain[f"sample_{suffix}_bucket"]   = chain.city_indicator.map(city_bucket)
    chain[f"sample_{suffix}_value"]    = chain.city_indicator.map(city_value)
    chain[f"sample_{suffix}_capacity"] = chain[f"sample_{suffix}_bucket"].map(QUINT)

# 9. Final column order, deduplicate, and save
chain["country_code"] = "CL"

core_cols = [
    "action_id", "global_mitigation_option", "action_mapping_strength", "strength_weight",
    "option_family", "feasibility_dimension", "global_indicator",
    "global_verdict_code", "global_verdict_description",
    "city_indicator", "country_code", "city_indicator_direction", "city_family_scope",
    "interpretation",
]
sample_cols = [c for c in chain.columns if c.startswith("sample_")]
chain = chain[core_cols + sample_cols].drop_duplicates().reset_index(drop=True)

out_path = DATA / "scoring_chain.csv"
chain.to_csv(out_path, index=False)

# Also write the legacy 12-column schema consumed by the DB chain-table importer.
# Drops the new strength_weight + country_code columns so the importer's column
# list stays stable across releases. Keep these two files in lockstep.
LEGACY_CHAIN_COLS = [
    "action_id", "global_mitigation_option", "action_mapping_strength",
    "option_family", "feasibility_dimension", "global_indicator",
    "global_verdict_code", "global_verdict_description",
    "city_indicator", "city_indicator_direction", "city_family_scope",
    "interpretation",
]
legacy_path = OUT / "scoring_chain_full.csv"
chain[LEGACY_CHAIN_COLS].to_csv(legacy_path, index=False)

active_bridges = chain["city_indicator"].notna().sum()
no_bridge_rows = chain["city_indicator"].isna().sum()
print(f"Wrote {out_path.name}: {len(chain)} rows")
print(f"Wrote {legacy_path.name} (legacy 12-col schema for DB import): {len(chain)} rows")
print(f"  actions covered:    {chain.action_id.nunique()}")
print(f"  IPCC options:       {chain.global_mitigation_option.nunique()}")
print(f"  active bridges:     {active_bridges} rows (city_indicator populated)")
print(f"  no-bridge cells:    {no_bridge_rows} rows (city columns NULL; explanation in interpretation)")
print()
print("Sample — chain for action icare_0117 (Public transport), first 10 rows:")
chain[chain.action_id == "icare_0117"].head(10)


Wrote scoring_chain.csv: 1900 rows
Wrote scoring_chain_full.csv (legacy 12-col schema for DB import): 1900 rows
  actions covered:    86
  IPCC options:       17
  active bridges:     426 rows (city_indicator populated)
  no-bridge cells:    1474 rows (city columns NULL; explanation in interpretation)

Sample — chain for action icare_0117 (Public transport), first 10 rows:


,action_id,global_mitigation_option,action_mapping_strength,strength_weight,option_family,feasibility_dimension,global_indicator,global_verdict_code,global_verdict_description,city_indicator,country_code,city_indicator_direction,city_family_scope,interpretation
1119,icare_0117,Public transport,direct,1.0,transport,economic,cost-effectiveness,C,supportive,median_household_income,CL,positive,all,Cities with higher median_household_income hav...
1120,icare_0117,Public transport,direct,1.0,transport,economic,cost-effectiveness,C,supportive,poverty_rate,CL,negative,all,Cities with higher poverty_rate have weaker fe...
1121,icare_0117,Public transport,direct,1.0,transport,economic,distributional_effects,C,supportive,poverty_rate,CL,negative,all,Cities with higher poverty_rate have weaker fe...
1122,icare_0117,Public transport,direct,1.0,transport,economic,employment_productivity,C,supportive,unemployment_rate,CL,positive,all,Cities with higher unemployment_rate have stro...
1123,icare_0117,Public transport,direct,1.0,transport,economic,employment_productivity,C,supportive,employment_in_transport_and_logistics,CL,positive,transport,Cities with higher employment_in_transport_and...
1124,icare_0117,Public transport,direct,1.0,transport,technological,technical_scalability,C,supportive,NaN,CL,NaN,NaN,IPCC verdict is 'supportive' but no city-level...
1125,icare_0117,Public transport,direct,1.0,transport,technological,maturity,C,supportive,NaN,CL,NaN,NaN,IPCC verdict is 'supportive' but no city-level...
1126,icare_0117,Public transport,direct,1.0,transport,technological,simplicity,B,neutral,NaN,CL,NaN,NaN,IPCC verdict is 'neutral' for this indicator —...
1127,icare_0117,Public transport,direct,1.0,transport,technological,absence_of_risk,C,supportive,NaN,CL,NaN,NaN,IPCC verdict is 'supportive' but no city-level...
1128,icare_0117,Public transport,direct,1.0,transport,institutional,political_acceptability,C,supportive,NaN,CL,NaN,NaN,IPCC verdict is 'supportive' but no city-level...
